# Air Quality Silver Layer

Transforms Open-Meteo Air Quality API responses from `weather_api.bronze.air_quality_raw` into clean, typed, analytics-ready Silver tables.

**Tables created:**
- `weather_api.silver.air_quality_current` — One row per location per API snapshot (grain: `location_name` + `snapshot_at`)
- `weather_api.silver.air_quality_hourly` — One row per location, forecast hour, and API snapshot (grain: `location_name` + `snapshot_at` + `forecast_for`)

**Units:** All pollutant concentrations in μg/m³, pollen in grains/m³, AOD dimensionless, UV index dimensionless, AQI indices are composite scores.

**Processing:** Batch read from Bronze → transform → Delta MERGE into Silver. Idempotent and safe for reruns.

In [0]:
# ════════════════════════════════════════════════════════════════════
# AIR QUALITY SILVER — TRANSFORM
# ════════════════════════════════════════════════════════════════════
from pyspark.sql.functions import (
    col, explode, to_timestamp, arrays_zip, count as cnt,
    countDistinct, desc, max as max_fn, min as min_fn, size, lit
)
from delta.tables import DeltaTable

# ─── Schema & Config ────────────────────────────────────────────────
spark.sql("CREATE SCHEMA IF NOT EXISTS weather_api.silver")

BRONZE_TABLE     = "weather_api.bronze.air_quality_raw"
SILVER_CURRENT   = "weather_api.silver.air_quality_current"
SILVER_HOURLY    = "weather_api.silver.air_quality_hourly"

# ─── Read Bronze Source ────────────────────────────────────────────
bronze = spark.table(BRONZE_TABLE)
total_bronze = bronze.count()
print(f"Bronze rows: {total_bronze}")

# ── Location ID lineage (backward-compatible) ──
_metadata_fields = [f for f in bronze.schema.fields if f.name == "metadata"][0].dataType.fields
_has_location_id = "location_id" in [f.name for f in _metadata_fields]
if _has_location_id:
    _location_id_expr = col("metadata.location_id")
else:
    _location_id_expr = lit(None).cast("long").alias("location_id")
    print("⚠️  metadata.location_id not found in Bronze schema — using NULL placeholder")

# ─── Transform: Current Observations ───────────────────────────────
# One row per location per API snapshot. Extracts the real-time air
# quality readings from raw_response.current and flattens metadata.

# Pollen and ammonia fields are string-typed in Bronze (often null due
# to seasonality). We cast them to double; nulls are preserved.
silver_current = bronze.select(
    # Lineage
    col("metadata.location_name"),
    _location_id_expr,
    col("metadata.latitude"),
    col("metadata.longitude"),
    col("raw_response.elevation").alias("elevation_m"),
    to_timestamp(col("metadata.retrieval_timestamp_utc")).alias("snapshot_at"),
    to_timestamp(col("raw_response.current.time")).alias("observed_at"),
    col("metadata.source"),
    col("metadata.api_endpoint"),
    col("_source_file_path").alias("source_file_path"),
    col("_ingest_timestamp").alias("ingested_at"),
    col("_rescued_data"),
    # Pollutants (μg/m³) — already double in Bronze
    col("raw_response.current.pm2_5"),
    col("raw_response.current.pm10"),
    col("raw_response.current.carbon_monoxide"),
    col("raw_response.current.nitrogen_dioxide"),
    col("raw_response.current.sulphur_dioxide"),
    col("raw_response.current.ozone"),
    col("raw_response.current.dust"),
    col("raw_response.current.aerosol_optical_depth"),
    # AQI indices (composite scores, cast to int for consistency with hourly)
    col("raw_response.current.european_aqi").cast("int").alias("european_aqi"),
    col("raw_response.current.us_aqi").cast("int").alias("us_aqi"),
    col("raw_response.current.us_aqi_carbon_monoxide").cast("int").alias("us_aqi_carbon_monoxide"),
    col("raw_response.current.us_aqi_nitrogen_dioxide").cast("int").alias("us_aqi_nitrogen_dioxide"),
    col("raw_response.current.us_aqi_ozone").cast("int").alias("us_aqi_ozone"),
    col("raw_response.current.us_aqi_pm10").cast("int").alias("us_aqi_pm10"),
    col("raw_response.current.us_aqi_pm2_5").cast("int").alias("us_aqi_pm2_5"),
    col("raw_response.current.us_aqi_sulphur_dioxide").cast("int").alias("us_aqi_sulphur_dioxide"),
    # Measurement interval (seconds the current reading represents)
    col("raw_response.current.interval").cast("int").alias("interval_seconds"),
    # UV index (dimensionless)
    col("raw_response.current.uv_index"),
    col("raw_response.current.uv_index_clear_sky"),
    # Pollen & ammonia (string in Bronze → cast to double, nulls preserved)
    col("raw_response.current.alder_pollen").cast("double").alias("alder_pollen"),
    col("raw_response.current.ammonia").cast("double").alias("ammonia"),
    col("raw_response.current.birch_pollen").cast("double").alias("birch_pollen"),
    col("raw_response.current.grass_pollen").cast("double").alias("grass_pollen"),
    col("raw_response.current.mugwort_pollen").cast("double").alias("mugwort_pollen"),
    col("raw_response.current.olive_pollen").cast("double").alias("olive_pollen"),
    col("raw_response.current.ragweed_pollen").cast("double").alias("ragweed_pollen"),
)

current_count = silver_current.count()
print(f"Current rows: {current_count}")

# ─── Transform: Hourly Forecast ────────────────────────────────────
# Explodes the hourly arrays into one row per location, forecast hour,
# and snapshot. Uses arrays_zip to align all measurement arrays with the
# time array by position, then explodes.

# Hourly field specs: (bronze_name, silver_name, spark_type)
HOURLY_SPECS = [
    ("time",                       "forecast_for",                "timestamp"),
    ("pm2_5",                      "pm2_5",                      "double"),
    ("pm10",                       "pm10",                       "double"),
    ("carbon_monoxide",            "carbon_monoxide",            "double"),
    ("nitrogen_dioxide",           "nitrogen_dioxide",           "double"),
    ("sulphur_dioxide",            "sulphur_dioxide",            "double"),
    ("ozone",                      "ozone",                      "double"),
    ("dust",                       "dust",                       "double"),
    ("aerosol_optical_depth",      "aerosol_optical_depth",      "double"),
    ("european_aqi",               "european_aqi",               "int"),
    ("us_aqi",                     "us_aqi",                     "int"),
    ("us_aqi_carbon_monoxide",     "us_aqi_carbon_monoxide",     "int"),
    ("us_aqi_nitrogen_dioxide",    "us_aqi_nitrogen_dioxide",    "int"),
    ("us_aqi_ozone",               "us_aqi_ozone",               "int"),
    ("us_aqi_pm10",                "us_aqi_pm10",                "int"),
    ("us_aqi_pm2_5",               "us_aqi_pm2_5",               "int"),
    ("us_aqi_sulphur_dioxide",     "us_aqi_sulphur_dioxide",     "int"),
    ("uv_index",                   "uv_index",                   "double"),
    ("uv_index_clear_sky",         "uv_index_clear_sky",         "double"),
    ("alder_pollen",               "alder_pollen",               "double"),
    ("ammonia",                    "ammonia",                    "double"),
    ("birch_pollen",               "birch_pollen",               "double"),
    ("grass_pollen",               "grass_pollen",               "double"),
    ("mugwort_pollen",             "mugwort_pollen",             "double"),
    ("olive_pollen",               "olive_pollen",               "double"),
    ("ragweed_pollen",             "ragweed_pollen",             "double"),
]

def _hourly_select(spec):
    """Build a select expression for one hourly field after arrays_zip."""
    bronze_name, silver_name, dtype = spec
    base = col(f"hourly.{bronze_name}")
    if dtype == "timestamp":
        return to_timestamp(base).alias(silver_name)
    elif dtype == "int":
        return base.cast("int").alias(silver_name)
    else:
        return base.cast("double").alias(silver_name)

# Zip all hourly arrays into an array of structs, then explode
hourly_zip_cols = [col(f"raw_response.hourly.{s[0]}") for s in HOURLY_SPECS]

silver_hourly = bronze.select(
    # Lineage
    col("metadata.location_name"),
    _location_id_expr,
    col("metadata.latitude"),
    col("metadata.longitude"),
    col("raw_response.elevation").alias("elevation_m"),
    to_timestamp(col("metadata.retrieval_timestamp_utc")).alias("snapshot_at"),
    col("metadata.source"),
    col("metadata.api_endpoint"),
    col("_source_file_path").alias("source_file_path"),
    col("_ingest_timestamp").alias("ingested_at"),
    col("_rescued_data"),
    # Explode zipped hourly arrays
    explode(arrays_zip(*hourly_zip_cols)).alias("hourly"),
).select(
    # Lineage passthrough
    "location_name", "latitude", "longitude", "elevation_m",
    "snapshot_at", "source", "api_endpoint", "source_file_path",
    "ingested_at", "_rescued_data",
    # Hourly measurement fields
    *[_hourly_select(s) for s in HOURLY_SPECS],
)

hourly_count = silver_hourly.count()
print(f"Hourly rows: {hourly_count}")

# ─── Validation ────────────────────────────────────────────────────
# Check for nulls in required fields
print("\n--- Current: Null counts ---")
silver_current.agg(
    cnt("*").alias("total"),
    cnt("location_name").alias("loc_nn"),
    cnt("snapshot_at").alias("snap_nn"),
    cnt("observed_at").alias("obs_nn"),
    cnt("pm2_5").alias("pm25_nn"),
    cnt("us_aqi").alias("aqi_nn"),
    cnt("alder_pollen").alias("alder_nn"),
).show(truncate=False)

# Verify hourly array alignment: all rows should have forecast_for
print("--- Hourly: Null counts ---")
silver_hourly.agg(
    cnt("*").alias("total"),
    cnt("forecast_for").alias("forecast_nn"),
    cnt("pm2_5").alias("pm25_nn"),
).show(truncate=False)

# Reconciliation: expect 64 current rows and 64×24=1536 hourly rows
expected_hourly = total_bronze * 24
print(f"\nExpected current rows: {total_bronze}, actual: {current_count}")
print(f"Expected hourly rows: {expected_hourly}, actual: {hourly_count}")
assert current_count == total_bronze, "Current row count mismatch!"
assert hourly_count == expected_hourly, "Hourly row count mismatch!"

# Check for duplicate keys
current_dups = silver_current.groupBy("location_name", "snapshot_at").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"Current duplicate keys: {current_dups.count()}")

hourly_dups = silver_hourly.groupBy("location_name", "snapshot_at", "forecast_for").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"Hourly duplicate keys: {hourly_dups.count()}")

# Check _rescued_data
rescued = bronze.filter(col("_rescued_data").isNotNull()).count()
print(f"\nRescued data records: {rescued} (0 = no schema drift)")

print("\n✓ Transform validation passed.")

In [0]:
# ════════════════════════════════════════════════════════════════════
# AIR QUALITY SILVER — WRITE & VALIDATE
# ════════════════════════════════════════════════════════════════════
# Uses Delta MERGE for idempotent upserts. Safe for reruns: existing
# rows are updated, new rows are inserted, no duplicates created.

# ── Add location_id column to existing tables (backward-compatible) ──
for _tbl in [SILVER_CURRENT, SILVER_HOURLY]:
    if spark.catalog.tableExists(_tbl):
        _has_loc_id = spark.sql(f"DESCRIBE TABLE {_tbl}").filter("col_name = 'location_id'").count() > 0
        if not _has_loc_id:
            spark.sql(f"ALTER TABLE {_tbl} ADD COLUMN location_id BIGINT")
            print(f"Added location_id column to {_tbl}")

# ─── Write: air_quality_current ────────────────────────────────────
# Schema fix: drop if column types are stale (bigint → int) or interval_seconds missing
if spark.catalog.tableExists(SILVER_CURRENT):
    _cols = spark.sql(f"DESCRIBE TABLE {SILVER_CURRENT}")
    _eaqi_type = _cols.filter("col_name = 'european_aqi'").collect()[0].data_type
    _has_interval = _cols.filter("col_name = 'interval_seconds'").count() > 0
    if _eaqi_type == "bigint" or not _has_interval:
        spark.sql(f"DROP TABLE {SILVER_CURRENT}")
        print(f"Dropped {SILVER_CURRENT} for type standardization + interval_seconds")

if spark.catalog.tableExists(SILVER_CURRENT):
    DeltaTable.forName(spark, SILVER_CURRENT).alias("t").merge(
        silver_current.alias("s"),
        "t.location_name = s.location_name AND t.snapshot_at = s.snapshot_at"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"MERGE completed: {SILVER_CURRENT}")
else:
    silver_current.write.format("delta").saveAsTable(SILVER_CURRENT)
    print(f"Created: {SILVER_CURRENT}")

# ─── Write: air_quality_hourly ─────────────────────────────────────
if spark.catalog.tableExists(SILVER_HOURLY):
    DeltaTable.forName(spark, SILVER_HOURLY).alias("t").merge(
        silver_hourly.alias("s"),
        "t.location_name = s.location_name AND t.snapshot_at = s.snapshot_at "
        "AND t.forecast_for = s.forecast_for"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
    print(f"MERGE completed: {SILVER_HOURLY}")
else:
    silver_hourly.write.format("delta").saveAsTable(SILVER_HOURLY)
    print(f"Created: {SILVER_HOURLY}")

# ─── Post-Write Validation ─────────────────────────────────────────
print("\n" + "=" * 70)
print("POST-WRITE VALIDATION")
print("=" * 70)

# Current table
curr = spark.table(SILVER_CURRENT)
curr_rows = curr.count()
curr_keys = curr.select(countDistinct("location_name", "snapshot_at")).collect()[0][0]
print(f"\n[{SILVER_CURRENT}]")
print(f"  Rows: {curr_rows}")
print(f"  Distinct (location_name, snapshot_at): {curr_keys}")
curr_dups = curr.groupBy("location_name", "snapshot_at").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"  Duplicate keys: {curr_dups.count()}")
assert curr_rows == curr_keys, "Duplicate keys in current table!"

print("  Null counts:")
curr.agg(
    cnt("*").alias("total"),
    cnt("pm2_5").alias("pm25_nn"),
    cnt("pm10").alias("pm10_nn"),
    cnt("ozone").alias("ozone_nn"),
    cnt("us_aqi").alias("us_aqi_nn"),
    cnt("european_aqi").alias("eaqi_nn"),
    cnt("uv_index").alias("uv_nn"),
    cnt("interval_seconds").alias("interval_nn"),
    cnt("alder_pollen").alias("alder_nn"),
).show(truncate=False)

print("  Timestamp ranges:")
curr.agg(
    min_fn("snapshot_at").alias("min_snapshot"),
    max_fn("snapshot_at").alias("max_snapshot"),
    min_fn("observed_at").alias("min_observed"),
    max_fn("observed_at").alias("max_observed"),
).show(truncate=False)

# Hourly table
hrly = spark.table(SILVER_HOURLY)
hrly_rows = hrly.count()
hrly_keys = hrly.select(countDistinct("location_name", "snapshot_at", "forecast_for")).collect()[0][0]
print(f"\n[{SILVER_HOURLY}]")
print(f"  Rows: {hrly_rows}")
print(f"  Distinct (location_name, snapshot_at, forecast_for): {hrly_keys}")
hrly_dups = hrly.groupBy("location_name", "snapshot_at", "forecast_for").agg(cnt("*").alias("n")).filter(col("n") > 1)
print(f"  Duplicate keys: {hrly_dups.count()}")
assert hrly_rows == hrly_keys, "Duplicate keys in hourly table!"

print("  Null counts:")
hrly.agg(
    cnt("*").alias("total"),
    cnt("forecast_for").alias("forecast_nn"),
    cnt("pm2_5").alias("pm25_nn"),
    cnt("ozone").alias("ozone_nn"),
).show(truncate=False)

print("  Timestamp ranges:")
hrly.agg(
    min_fn("forecast_for").alias("min_forecast"),
    max_fn("forecast_for").alias("max_forecast"),
).show(truncate=False)

# Reconciliation
print("\n--- Reconciliation ---")
print(f"  Bronze rows: {total_bronze}")
print(f"  Current rows: {curr_rows} (expected: {total_bronze})")
print(f"  Hourly rows: {hrly_rows} (expected: {total_bronze * 24})")
print(f"  Rescued data: {rescued}")
assert curr_rows == total_bronze, "Current row count mismatch!"
assert hrly_rows == total_bronze * 24, "Hourly row count mismatch!"

# Sample rows
print("\n--- Sample current rows ---")
curr.select("location_name", "snapshot_at", "observed_at", "pm2_5", "pm10", "ozone", "us_aqi", "uv_index", "interval_seconds").show(5, truncate=False)

print("\n--- Sample hourly rows ---")
hrly.select("location_name", "snapshot_at", "forecast_for", "pm2_5", "ozone", "us_aqi").orderBy("location_name", "forecast_for").show(5, truncate=False)

print("\n✓ Air Quality Silver layer complete.")

In [0]:
# ════════════════════════════════════════════════════════════════════
# AIR QUALITY SILVER — COLUMN COMMENTS
# ════════════════════════════════════════════════════════════════════
# Idempotent: ALTER COLUMN COMMENT can be run multiple times safely.

_comments = {
    "weather_api.silver.air_quality_current": {
        "location_name": "City name used as the location identifier",
        "latitude": "Requested latitude coordinate",
        "longitude": "Requested longitude coordinate",
        "elevation_m": "Elevation of the location, in metres above sea level",
        "snapshot_at": "UTC timestamp when the source API snapshot was retrieved",
        "observed_at": "UTC timestamp the current air quality reading applies to",
        "source": "Pipeline source label",
        "api_endpoint": "Full API URL called for this observation",
        "source_file_path": "UC Volume path of the source JSON file",
        "ingested_at": "Timestamp when Auto Loader ingested this record into Bronze",
        "_rescued_data": "Fields that did not match the Bronze schema (null = no schema drift)",
        "pm2_5": "Fine particulate matter (≤2.5 µm) concentration, in µg/m³",
        "pm10": "Coarse particulate matter (≤10 µm) concentration, in µg/m³",
        "carbon_monoxide": "Carbon monoxide (CO) concentration, in µg/m³",
        "nitrogen_dioxide": "Nitrogen dioxide (NO₂) concentration, in µg/m³",
        "sulphur_dioxide": "Sulphur dioxide (SO₂) concentration, in µg/m³",
        "ozone": "Ozone (O₃) concentration, in µg/m³",
        "dust": "Suspended particulate matter / dust mass, in µg/m³",
        "aerosol_optical_depth": "Aerosol optical depth (light scattering/absorption by particles)",
        "european_aqi": "European Air Quality Index (0=good, 100=poor)",
        "us_aqi": "US Air Quality Index (0–50 good, 51–100 moderate, 101+ unhealthy)",
        "us_aqi_carbon_monoxide": "US AQI sub-index for carbon monoxide",
        "us_aqi_nitrogen_dioxide": "US AQI sub-index for nitrogen dioxide",
        "us_aqi_ozone": "US AQI sub-index for ozone",
        "us_aqi_pm10": "US AQI sub-index for PM10",
        "us_aqi_pm2_5": "US AQI sub-index for PM2.5",
        "us_aqi_sulphur_dioxide": "US AQI sub-index for sulphur dioxide",
        "interval_seconds": "Time interval the current reading represents, in seconds (3600 = hourly average)",
        "uv_index": "Solar ultraviolet radiation level (0=none, 11+=extreme)",
        "uv_index_clear_sky": "UV index assuming clear sky conditions",
        "alder_pollen": "Alder pollen concentration, in grains/m³ (seasonal, may be null)",
        "ammonia": "Ammonia (NH₃) concentration, in µg/m³ (may be null)",
        "birch_pollen": "Birch pollen concentration, in grains/m³ (seasonal, may be null)",
        "grass_pollen": "Grass pollen concentration, in grains/m³ (seasonal, may be null)",
        "mugwort_pollen": "Mugwort pollen concentration, in grains/m³ (seasonal, may be null)",
        "olive_pollen": "Olive pollen concentration, in grains/m³ (seasonal, may be null)",
        "ragweed_pollen": "Ragweed pollen concentration, in grains/m³ (seasonal, may be null)",
    },
    "weather_api.silver.air_quality_hourly": {
        "location_name": "City name used as the location identifier",
        "latitude": "Requested latitude coordinate",
        "longitude": "Requested longitude coordinate",
        "elevation_m": "Elevation of the location, in metres above sea level",
        "snapshot_at": "UTC timestamp when the source API snapshot was retrieved",
        "source": "Pipeline source label",
        "api_endpoint": "Full API URL called for this forecast",
        "source_file_path": "UC Volume path of the source JSON file",
        "ingested_at": "Timestamp when Auto Loader ingested this record into Bronze",
        "_rescued_data": "Fields that did not match the Bronze schema (null = no schema drift)",
        "forecast_for": "UTC timestamp for this forecast hour",
        "pm2_5": "Fine particulate matter (≤2.5 µm) concentration, in µg/m³",
        "pm10": "Coarse particulate matter (≤10 µm) concentration, in µg/m³",
        "carbon_monoxide": "Carbon monoxide (CO) concentration, in µg/m³",
        "nitrogen_dioxide": "Nitrogen dioxide (NO₂) concentration, in µg/m³",
        "sulphur_dioxide": "Sulphur dioxide (SO₂) concentration, in µg/m³",
        "ozone": "Ozone (O₃) concentration, in µg/m³",
        "dust": "Suspended particulate matter / dust mass, in µg/m³",
        "aerosol_optical_depth": "Aerosol optical depth (light scattering/absorption by particles)",
        "european_aqi": "European Air Quality Index (0=good, 100=poor)",
        "us_aqi": "US Air Quality Index (0–50 good, 51–100 moderate, 101+ unhealthy)",
        "us_aqi_carbon_monoxide": "US AQI sub-index for carbon monoxide",
        "us_aqi_nitrogen_dioxide": "US AQI sub-index for nitrogen dioxide",
        "us_aqi_ozone": "US AQI sub-index for ozone",
        "us_aqi_pm10": "US AQI sub-index for PM10",
        "us_aqi_pm2_5": "US AQI sub-index for PM2.5",
        "us_aqi_sulphur_dioxide": "US AQI sub-index for sulphur dioxide",
        "uv_index": "Solar ultraviolet radiation level (0=none, 11+=extreme)",
        "uv_index_clear_sky": "UV index assuming clear sky conditions",
        "alder_pollen": "Alder pollen concentration, in grains/m³ (seasonal, may be null)",
        "ammonia": "Ammonia (NH₃) concentration, in µg/m³ (may be null)",
        "birch_pollen": "Birch pollen concentration, in grains/m³ (seasonal, may be null)",
        "grass_pollen": "Grass pollen concentration, in grains/m³ (seasonal, may be null)",
        "mugwort_pollen": "Mugwort pollen concentration, in grains/m³ (seasonal, may be null)",
        "olive_pollen": "Olive pollen concentration, in grains/m³ (seasonal, may be null)",
        "ragweed_pollen": "Ragweed pollen concentration, in grains/m³ (seasonal, may be null)",
    },
}

_comment_count = 0
for _table, _col_comments in _comments.items():
    for _col, _comment in _col_comments.items():
        _escaped = _comment.replace("'", "''")
        spark.sql(f"ALTER TABLE {_table} ALTER COLUMN {_col} COMMENT '{_escaped}'")
        _comment_count += 1
print(f"Applied {_comment_count} column comments across {len(_comments)} air quality Silver tables.")